In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Load saved embeddings + labels
X = np.load("albert_embeddings.npy")
y = np.load("albert_labels.npy")

# Train-test split (stratify removed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Get unique classes - make sure we only have 0-17
classes = np.unique(y)
print(f"Unique classes found: {classes}")
print(f"Number of classes: {len(classes)}")

# Verify we have classes 0-21
expected_classes = np.arange(22)
missing_classes = set(expected_classes) - set(classes)
extra_classes = set(classes) - set(expected_classes)

if missing_classes:
    print(f"Warning: Missing classes: {missing_classes}")
if extra_classes:
    print(f"Warning: Extra classes found: {extra_classes}")
    # Filter out any class 22 or higher
    mask = y < 22
    X = X[mask]
    y = y[mask]
    print(f"Filtered dataset to remove classes >= 22")
    # Re-split after filtering
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    classes = np.unique(y)

# Sort classes to ensure 0-17 order
classes = np.sort(classes)
n_classes = len(classes)
print(f"Final classes: {classes}")
print(f"Final number of classes: {n_classes}")

# Specificity (macro)
def specificity_macro(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n = cm.shape[0]
    specs = []
    for i in range(n):
        TN = cm.sum() - (cm[i].sum() + cm[:, i].sum() - cm[i, i])
        FP = cm[:, i].sum() - cm[i, i]
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0)
    return np.mean(specs)

# Models
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=150, random_state=42),
    "SVM": SVC(kernel="linear", probability=True, random_state=42)  # Enable probability for ROC
}

# Optional models
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=200, n_jobs=-1, random_state=42)
except:
    pass

# Create figure for ROC curves comparison
fig_combined, axes_combined = plt.subplots(2, 2, figsize=(16, 14))
axes_combined = axes_combined.ravel()

# Train + Evaluate
model_count = 0
for name, model in models.items():
    print("\n" + "="*60)
    print(" " * 20 + f"{name}")
    print("="*60)
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    spec = specificity_macro(y_test, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision (macro): {prec:.4f}")
    print(f"Recall (macro): {rec:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"Specificity (macro): {spec:.4f}")
    
    # ==================== CONFUSION MATRIX PLOT ====================
    cm = confusion_matrix(y_test, y_pred, labels=classes)
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count'},
                square=True)
    plt.title(f'Confusion Matrix - {name}\n(22 Classes: 0-21)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Predicted Labels', fontsize=14)
    plt.ylabel('True Labels', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    
    # Add accuracy in the plot
    plt.figtext(0.5, 0.01, 
                f'Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}', 
                ha='center', fontsize=12, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.7))
    
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{name.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Confusion matrix saved as 'confusion_matrix_{name.lower().replace(' ', '_')}.png'")
    
    # ==================== ROC CURVE FOR EACH CLASS ====================
    print(f"\nROC Curve Analysis for {name}:")
    
    # Get probability scores for ROC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
        if y_score.ndim == 1:
            y_score = y_score.reshape(-1, 1)
    else:
        print(f"  ⚠️ {name} doesn't support probability scores. Skipping ROC analysis.")
        continue
    
    # Binarize the test labels for multiclass ROC
    y_test_bin = label_binarize(y_test, classes=classes)
    
    # Compute ROC curve and ROC area for each class
    fpr = {}
    tpr = {}
    roc_auc = {}
    
    # Make sure we only process the classes we have (0-17)
    for i, class_label in enumerate(classes):
        if y_score.shape[1] > i:
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
            print(f"  Class {class_label:2d}: AUC = {roc_auc[i]:.4f}")
    
    # Plot ROC curves for all classes in one figure
    plt.figure(figsize=(16, 14))
    
    # Create a color palette for 22 classes
    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    
    for i, class_label in enumerate(classes):
        if i in fpr:  # Check if ROC was computed for this class
            plt.plot(fpr[i], tpr[i], color=colors[i], lw=2.5, alpha=0.8,
                    label=f'Class {class_label:2d} (AUC = {roc_auc[i]:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.6, label='Random (AUC = 0.500)')
    
    # Calculate micro-average ROC
    if y_score.ndim > 1 and y_score.shape[1] > 1:
        fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
        roc_auc_micro = auc(fpr_micro, tpr_micro)
        plt.plot(fpr_micro, tpr_micro, color='black', lw=3, linestyle=':',
                label=f'Micro-average (AUC = {roc_auc_micro:.3f})')
        print(f"  Micro-average AUC: {roc_auc_micro:.4f}")
    
    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'ROC Curves for All {n_classes} Classes (0-{n_classes-1}) - {name}', 
              fontsize=16, fontweight='bold', pad=20)
    
    # Create legend with multiple columns
    plt.legend(loc='lower right', fontsize=9, ncol=2, frameon=True, 
               framealpha=0.9, title=f'Classes (0-{n_classes-1})')
    
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.axhline(y=0, color='k', alpha=0.3)
    plt.axvline(x=0, color='k', alpha=0.3)
    plt.axhline(y=1, color='k', alpha=0.3)
    plt.axvline(x=1, color='k', alpha=0.3)
    
    # Add model performance metrics to ROC plot
    plt.figtext(0.5, 0.01, f'Accuracy: {acc:.4f} | F1-Score: {f1:.4f}', 
                ha='center', fontsize=12, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.7))
    
    plt.tight_layout()
    plt.savefig(f'roc_curve_all_classes_{name.lower().replace(" ", "_")}.png', 
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ ROC curve saved as 'roc_curve_all_classes_{name.lower().replace(' ', '_')}.png'")
    
    # ==================== COMBINED ROC PLOT (All models comparison) ====================
    if model_count < len(axes_combined):
        ax = axes_combined[model_count]
        
        # Plot ROC for each class
        for i, class_label in enumerate(classes):
            if i in fpr:  # Check if ROC was computed for this class
                ax.plot(fpr[i], tpr[i], color=colors[i], lw=1.5, alpha=0.6)
        
        # Add micro-average if available
        if 'roc_auc_micro' in locals():
            ax.plot(fpr_micro, tpr_micro, color='black', lw=2.5, 
                   linestyle='--', label=f'Micro-avg (AUC = {roc_auc_micro:.3f})')
        
        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.set_title(f'{name}\nAcc: {acc:.3f}, F1: {f1:.3f}', fontsize=12, fontweight='bold')
        ax.set_xlabel('False Positive Rate', fontsize=10)
        ax.set_ylabel('True Positive Rate', fontsize=10)
        ax.grid(True, alpha=0.2)
        ax.legend(loc='lower right', fontsize=8)
        
        model_count += 1
    
    print("-"*60)

# Save the combined ROC comparison figure
if model_count > 0:
    # Remove unused subplots
    for i in range(model_count, len(axes_combined)):
        fig_combined.delaxes(axes_combined[i])
    
    fig_combined.suptitle(f'ROC Curves Comparison - All Models (22 Classes: 0-21)', 
                         fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('roc_curves_comparison_all_models.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n✓ Combined ROC comparison saved as 'roc_curves_comparison_all_models.png'")

print("\n" + "="*60)
print("ALL VISUALIZATIONS COMPLETED")
print("="*60)
print("\nGenerated files:")
for name in models.keys():
    print(f"  • confusion_matrix_{name.lower().replace(' ', '_')}.png")
    print(f"  • roc_curve_all_classes_{name.lower().replace(' ', '_')}.png")
print("  • roc_curves_comparison_all_models.png")
print("\nEach confusion matrix shows exactly 22 classes (0-21).")
print("Each ROC curve shows all 22 classes with individual AUC scores.")

Unique classes found: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
Number of classes: 22
Final classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
Final number of classes: 22

                    RandomForest
Accuracy: 0.2472
Precision (macro): 0.2217
Recall (macro): 0.2480
F1-score (macro): 0.2295
Specificity (macro): 0.9642
✓ Confusion matrix saved as 'confusion_matrix_randomforest.png'

ROC Curve Analysis for RandomForest:
  Class  0: AUC = 0.4368
  Class  1: AUC = 0.8867
  Class  2: AUC = 0.8207
  Class  3: AUC = 0.5748
  Class  4: AUC = 0.7426
  Class  5: AUC = 0.7700
  Class  6: AUC = 0.8773
  Class  7: AUC = 0.8500
  Class  8: AUC = 0.7412
  Class  9: AUC = 0.6335
  Class 10: AUC = 0.8441
  Class 11: AUC = 0.7700
  Class 12: AUC = 0.9899
  Class 13: AUC = 0.6515
  Class 14: AUC = 0.6786
  Class 15: AUC = 0.7833
  Class 16: AUC = 0.9342
  Class 17: AUC = 0.8835
  Class 18: AUC = 0.7836
  Class 19: AUC = 0.7209
  Class 20: AUC = 0.74

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Accuracy: 0.1931
Precision (macro): 0.1672
Recall (macro): 0.1934
F1-score (macro): 0.1697
Specificity (macro): 0.9616
✓ Confusion matrix saved as 'confusion_matrix_adaboost.png'

ROC Curve Analysis for AdaBoost:
  Class  0: AUC = 0.5809
  Class  1: AUC = 0.8200
  Class  2: AUC = 0.7536
  Class  3: AUC = 0.5884
  Class  4: AUC = 0.6928
  Class  5: AUC = 0.6797
  Class  6: AUC = 0.7894
  Class  7: AUC = 0.7833
  Class  8: AUC = 0.7175
  Class  9: AUC = 0.6585
  Class 10: AUC = 0.7866
  Class 11: AUC = 0.6828
  Class 12: AUC = 0.9821
  Class 13: AUC = 0.6470
  Class 14: AUC = 0.6646
  Class 15: AUC = 0.6786
  Class 16: AUC = 0.8693
  Class 17: AUC = 0.7936
  Class 18: AUC = 0.7529
  Class 19: AUC = 0.7163
  Class 20: AUC = 0.7298
  Class 21: AUC = 0.6781
  Micro-average AUC: 0.7458
✓ ROC curve saved as 'roc_curve_all_classes_adaboost.png'
------------------------------------------------------------

                    SVM
Accuracy: 0.5201
Precision (macro): 0.5227
Recall (macro): 0.5198